# Step 12.5 — Inspect an Attention Matrix：怎样读懂一个 Tiny Causal `T×T` Matrix？

Step 12.4 已经有一个完整 single causal attention head。

现在不再引入新的 Attention 数学，只做最后一件事：

> **把 attention weights `A:(B,T,T)` 真正拿出来读，检查 causal structure 是否符合预期。**

本节会区分两类东西：

```text
Hand-computed expectation
→ 数值由我们事先设计，可以提前知道

Actual runtime output
→ 只有执行 Notebook 后才能看到具体数值
```

因为本轮只准备母版、不运行 Notebook，所以后者只写检查代码和预期结构，不声称已经获得实际运行结果。

本节完成即停，不进入 Step 13.1，也不进入 Multi-Head Attention。


## 1. 先固定读法：Rows 是 Queries，Columns 是 Keys

沿用 Step 11–12 的约定：

$$
A\in\mathbb R^{B\times T\times T}.
$$

对一个 sample，取：

$$
A^{(b)}\in\mathbb R^{T\times T}.
$$

其中：

```text
row i    = Query position i
column j = Key position j
A[i,j]   = Query i 给 Key j 的 attention weight
```

所以读一行，就是问：

> “这个 Query position 把自己的 100% attention 分给了哪些合法 Keys？”

在 causal self-attention 中必须满足：

$$
A_{i,j}=0\quad\text{for }j>i.
$$

也就是严格上三角必须全部为 0。

另外每一行 softmax 后都应该满足：

$$
\boxed{\sum_{j=0}^{T-1}A_{i,j}=1}.
$$

所以最基本的两个检查就是：

```text
1. row sums ≈ 1
2. strict upper triangle = 0
```


## 2. 一个可以完全手算的 Tiny Expected Matrix

为了只检查 causal mechanism，不掺入 Q/K 内容差异，先故意设置：

$$
S_{scaled}
=
\begin{bmatrix}
0&0&0&0\\
0&0&0&0\\
0&0&0&0\\
0&0&0&0
\end{bmatrix}.
$$

也就是所有 Query–Key pairs 在 mask 前 compatibility 完全相同。

加入 causal mask 后：

$$
S_{masked}
=
\begin{bmatrix}
0&-\infty&-\infty&-\infty\\
0&0&-\infty&-\infty\\
0&0&0&-\infty\\
0&0&0&0
\end{bmatrix}.
$$

因为每一行所有**合法** logits 都相等，softmax 会在合法位置上均匀分配。

因此手算预期：

$$
\boxed{
A_{expected}
=
\begin{bmatrix}
1&0&0&0\\
1/2&1/2&0&0\\
1/3&1/3&1/3&0\\
1/4&1/4&1/4&1/4
\end{bmatrix}
}.
$$

这张矩阵非常适合第一次读 causal attention：

- Query 0 只有 Key 0 合法，所以 `[1,0,0,0]`；
- Query 1 有两个合法 Keys，所以各 `1/2`；
- Query 2 有三个合法 Keys，所以各 `1/3`；
- Query 3 已经没有 future position，所以四个 Keys 都合法，各 `1/4`。

这里的均匀分配不是 causal attention 的一般规律，只是因为这个教学例子故意让所有合法 logits 相等。


In [ ]:
import mlx.core as mx

T = 4

# Designed teaching case: all pre-mask scaled scores are equal.
scaled_scores = mx.zeros((T, T))

future_upper = mx.triu(mx.ones((T, T)), k=1) > 0
causal_mask = mx.where(
    future_upper,
    float('-inf'),
    0.0,
)

masked_scores = scaled_scores + causal_mask
attention_weights = mx.softmax(masked_scores, axis=-1)

print('masked_scores:')
print(masked_scores)
print('attention_weights:')
print(attention_weights)
print('row sums:')
print(mx.sum(attention_weights, axis=-1))

# When YOU execute this work copy, the expected matrix is:
# [[1.0000, 0.0000, 0.0000, 0.0000],
#  [0.5000, 0.5000, 0.0000, 0.0000],
#  [0.3333, 0.3333, 0.3333, 0.0000],
#  [0.2500, 0.2500, 0.2500, 0.2500]]
# This notebook master has not been executed here.


## 3. 怎样一行一行读？

假设你执行上一个 cell 后得到与手算预期一致的矩阵：

```text
          Key 0   Key 1   Key 2   Key 3
Query 0   1.000    0       0       0
Query 1   0.500   0.500    0       0
Query 2   0.333   0.333   0.333    0
Query 3   0.250   0.250   0.250   0.250
```

### 读 Query 2

第三行：

$$
[1/3,1/3,1/3,0].
$$

含义不是：

> “token 2 与前三个 token 的语义一定一样相关。”

而只是这个**人工等-logit测试**里，Query 2 可以合法看 Keys 0、1、2，而且三者 pre-softmax scores 恰好相同。

最重要的 causal invariant 是最后一项：

$$
A_{2,3}=0.
$$

因为 Key 3 对 Query 2 来说属于 future。

### 读 Query 3

最后一行没有 future positions，所以 causal mask 没有强制任何位置归零。

但注意：真实训练好的 head 最后一行通常不会均匀 `[0.25,0.25,0.25,0.25]`；具体分布由 Q/K compatibility 决定。


## 4. 两个 Structural Checks：不要靠肉眼猜

### Check A — Row Sums

由于 softmax 沿 Key axis：

$$
\sum_j A_{i,j}=1.
$$

代码检查：

```python
row_sums = mx.sum(A, axis=-1)
```

如果是 batch：

$$
A:(B,T,T)
\rightarrow
row\_sums:(B,T).
$$

### Check B — Future Weights

future positions 是 strict upper triangle。

所以：

```python
future_weights = mx.triu(A, k=1)
```

理论上应该全为 0。

检查时可以看：

```python
mx.max(mx.abs(future_weights))
```

理想 causal `-inf` mask + softmax 下应为 0。

这里我们检查的是**结构性质**，不是要求某个合法 attention weight 必须取特定数值。


In [ ]:
# Structural checks for the deterministic teaching matrix above.
row_sums = mx.sum(attention_weights, axis=-1)
future_weights = mx.triu(attention_weights, k=1)
max_future_weight = mx.max(mx.abs(future_weights))

print('row sums:', row_sums)
print('strict-upper-triangle future weights:')
print(future_weights)
print('max future weight:', max_future_weight)

# Expected when executed:
# row sums are approximately [1, 1, 1, 1]
# max future weight is 0.
# These are expectations from the construction, not runtime results claimed here.


## 5. 承接 Step 12.4：怎样让真实 Head 暴露 Attention Weights 供诊断？

Step 12.4 的生产式教学版本只返回：

```python
return out
```

这是合理的，因为正常上层只需要 head output。

但为了 inspection，可以做一个 **diagnostic variant（诊断版本）**，额外返回：

```text
out
attention_weights
```

这不是改变 Attention 数学，只是让内部 tensor 暂时暴露给我们观察。

这里仍然使用同一个 causal pipeline：

```text
Q/K/V
→ scaled scores
→ causal mask
→ softmax
→ attention weights
→ A @ V
```

下面 diagnostic head 虽然 Q/K/V projection 权重是随机初始化的，但示例输入特意设为全零，并且三套 Linear 都使用 `bias=False`。因此无论随机权重是什么：

```text
x = 0
→ Q = K = V = 0
→ 所有未 mask 的 compatibility scores = 0
→ 每个 Query 在合法 Keys 上均匀分配 attention
→ future positions = 0
→ out = A @ V = 0
```

所以这个 diagnostic case 的合法位置权重其实**可以在运行前精确推导**，并且会与前面的 uniform causal matrix 相同。Notebook 仍未执行，因此这些是 mathematically expected values（数学预期值），不是本轮声称已经观测到的 runtime output。


In [ ]:
import math
import mlx.nn as nn


class InspectableCausalAttentionHead(nn.Module):
    def __init__(self, input_dims: int, head_dim: int):
        super().__init__()
        self.q_proj = nn.Linear(input_dims, head_dim, bias=False)
        self.k_proj = nn.Linear(input_dims, head_dim, bias=False)
        self.v_proj = nn.Linear(input_dims, head_dim, bias=False)
        self.scale = 1.0 / math.sqrt(head_dim)

    def __call__(self, x: mx.array):
        _, T, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        scores = (q @ mx.swapaxes(k, -1, -2)) * self.scale

        future_upper = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future_upper,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        masked_scores = scores + mask
        attention_weights = mx.softmax(masked_scores, axis=-1)
        out = attention_weights @ v

        return out, attention_weights


B, T, C = 1, 4, 6
head = InspectableCausalAttentionHead(input_dims=C, head_dim=3)
# Zero input + bias=False makes Q=K=V=0 regardless of random weights.
x = mx.zeros((B, T, C))

out, A = head(x)

print('out shape:', out.shape)
print('out:')
print(out)
print('attention shape:', A.shape)
print('attention matrix for sample 0:')
print(A[0])
print('row sums:', mx.sum(A[0], axis=-1))
print('future max:', mx.max(mx.abs(mx.triu(A[0], k=1))))

# IMPORTANT:
# This master notebook is unexecuted.
# Because x=0 and all three projections use bias=False, the exact expected
# legal-position pattern is deterministic: uniform over each allowed prefix,
# future weights are 0, and V=0 makes out exactly 0.
# These are mathematical expectations, not observed runtime results.


## 6. 可选 Heatmap：颜色只是 Weight 大小，结构仍靠坐标读

矩阵打印适合精确读数；heatmap 适合快速看模式。

横轴继续是：

```text
Key position j
```

纵轴继续是：

```text
Query position i
```

所以一个正确 causal attention heatmap 应该表现为：

> 主对角线以上的严格上三角区域全部是 0-weight 区域。

但是不要把某个亮块自动解释成人类语言学意义上的“模型正在理解某种关系”。

Attention weight 是模型内部 routing coefficient；具体语义解释需要额外证据。

下面代码只负责把执行时得到的 `A[0]` 画出来，并把数值标在格子里。本轮不运行它。


In [ ]:
import matplotlib.pyplot as plt

matrix = A[0]

fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(matrix, aspect='auto')

ax.set_xlabel('Key position j')
ax.set_ylabel('Query position i')
ax.set_title('Causal attention weights')
ax.set_xticks(range(T))
ax.set_yticks(range(T))

for i in range(T):
    for j in range(T):
        ax.text(j, i, f'{float(matrix[i, j]):.3f}', ha='center', va='center')

fig.colorbar(image, ax=ax, label='attention weight')
plt.show()


## 7. Step 12.5 到这里完成了什么？

现在 Step 12 的 causal chain 已经闭环：

```text
autoregressive next-token constraint
↓
j <= i visibility
↓
strict-upper-triangle causal mask
↓
stable masked softmax
↓
single causal attention head
↓
inspect T×T attention weights
↓
verify future weights = 0
```

本节最重要的检查不是“某个合法位置一定应该是多少”，而是两个 causal invariants：

$$
\boxed{\sum_j A_{i,j}=1}
$$

以及：

$$
\boxed{A_{i,j}=0\quad\text{for }j>i}.
$$

### Hand expectation 与 Runtime result 的边界

- 全零合法 logits 的教学例子可以提前手算出 exact expected pattern；
- 本节 diagnostic head 同样使用全零输入，且 Q/K/V projections 都是 `bias=False`，因此无论随机 weights 如何都有 `Q=K=V=0`：合法位置 attention 确定地在每行 causal prefix 上均匀分配，future weights 为 0，且 `out=0`；
- 本轮没有运行任何 cell，因此这些仍是数学预期值，不写成“实际观测输出”。

### Step 12.5 材料边界

- 行是 Query positions，列是 Key positions；每一行是一个 Query 对所有 Keys 的 distribution。
- softmax 使每行合法权重和为 1。
- causal mask 强制 strict upper triangle 的 future weights 为 0。
- hand-computed uniform example 只用于验证 mask/softmax 结构，不代表真实模型 attention 通常均匀。
- diagnostic head 可返回 attention weights 供执行时检查；在本节特意选择的 `x=0`、`bias=False` 诊断输入下，合法权重不依赖随机 Q/K 参数，而是确定地在每行合法 prefix 上均匀分配。
- 本轮只准备 Step 12.5；不进入 Step 13.1，不讨论为什么需要 multiple heads，也不实现 Multi-Head Attention。
